# M1-T02 — Colab smoke test, interruption and resume

Runner for card **[M1-T02]** of the arXiv corpus project (Group 10, CSC14119).
CPU-only Colab: nothing here needs a GPU.

What this notebook proves, in order:

1. Colab mounts Google Drive as the persistent store: `/content/drive/MyDrive/arxiv_corpus`.
2. `scripts/live_smoke.py` crawls `config/pilot_10.json` (10 papers), extracts `.tex`/`.bib`
   and drops every other file type.
3. The run is interrupted on purpose after paper 5. The batch attempt is recorded as
   `interrupted` and no temporary file survives.
4. A second invocation with `resume=True` skips the 5 finished papers (discovery cache,
   no re-download) and continues from paper 6.
5. `verify` re-hashes the corpus against contract v2 and `proof` prints the PASS/FAIL
   report to hand to Toan for the full-crawl sign-off.

**Request budget per fresh invocation** (with `--max-attempts 1`): at most 1 discovery +
N source requests per paper, where N is the number of versions arXiv reports. The resume
invocation only spends requests on the papers it still has to crawl.

**Before you start**: set the repo URL and (optionally) the Semantic Scholar key in the
next cell. A Drive folder named `arxiv_corpus` is created if missing.


In [ ]:
#@title 0. Parameters { display-mode: "form" }
REPO_URL = "https://github.com/Toan-ten/Arxiv-Project-Group10.git"  #@param {type:"string"}
REPO_DIR = "/content/Arxiv-Project-Group10"  #@param {type:"string"}
BRANCH = "main"  #@param {type:"string"}
DRIVE_BASE = "/content/drive/MyDrive/arxiv_corpus"  #@param {type:"string"}
BATCH_ID = "m1-smoke-pilot-10"  #@param {type:"string"}
PILOT = "config/pilot_10.json"  #@param {type:"string"}
INTERRUPT_AFTER = 5  #@param {type:"integer"}
USE_S2 = False  #@param {type:"boolean"}
BENCHMARK = True  #@param {type:"boolean"}
S2_MIN_INTERVAL = 2.0  #@param {type:"number"}

CORPUS_ROOT = f"{DRIVE_BASE}/corpus"
LOG_ROOT = f"{DRIVE_BASE}/logs"
REPORT_ROOT = f"{DRIVE_BASE}/reports"
print("corpus:", CORPUS_ROOT)
print("logs  :", LOG_ROOT)
print("reports:", REPORT_ROOT)
print("batch :", BATCH_ID)
print("benchmark:", BENCHMARK)


## 1. Environment check (CPU-only)

Confirms the interpreter and that no GPU is being used, so the run matches the M1
requirement to benchmark on a CPU-only runtime.


In [ ]:
import platform
import shutil
import sys

print("python  :", sys.version.split()[0], "|", platform.platform())
print("cpu     :", "CPU only (no nvidia-smi)" if shutil.which("nvidia-smi") is None else "GPU present")
print("disk    :", round(shutil.disk_usage("/content").free / 2**30, 1), "GiB free")

# The project targets Python 3.10+; Colab ships 3.10/3.11/3.12 depending on the image.
if sys.version_info < (3, 10):
    raise SystemExit("Python 3.10+ required")


## 2. Mount Google Drive

Drive is the persistent store, so an interrupted Colab session loses nothing: the corpus,
the batch logs and the attempts all live on the mounted disk rather than in `/content`.


In [ ]:
import os
from pathlib import Path

MOUNTED = False
try:
    from google.colab import drive

    try:
        drive.mount("/content/drive")
        MOUNTED = True
    except Exception as exc:  # already mounted in this session
        print("drive.mount said:", exc)
        MOUNTED = Path("/content/drive/MyDrive").is_dir()
except ImportError:
    print("Not running in Colab: using the local filesystem instead of Drive.")

Path(CORPUS_ROOT).mkdir(parents=True, exist_ok=True)
Path(LOG_ROOT).mkdir(parents=True, exist_ok=True)
print("mounted_drive:", MOUNTED)
print("corpus exists:", Path(CORPUS_ROOT).is_dir())
print("logs exists  :", Path(LOG_ROOT).is_dir())

if MOUNTED:
    usage = shutil.disk_usage("/content/drive")
    print("drive free   :", round(usage.free / 2**30, 1), "GiB")


## 3. Get the project

Clones on the first run and fast-forwards afterwards, so re-running the notebook after a
`git push` picks up the new code without losing the corpus on Drive.


In [ ]:
import subprocess

path = Path(REPO_DIR)
if (path / ".git").is_dir():
    print(subprocess.run(["git", "-C", REPO_DIR, "fetch", "--depth", "1",
                          "origin", BRANCH], text=True, capture_output=True).stdout)
    print(subprocess.run(["git", "-C", REPO_DIR, "checkout", BRANCH],
                         text=True, capture_output=True).stdout)
    print(subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"],
                         text=True, capture_output=True).stdout)
else:
    result = subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH, REPO_URL, REPO_DIR],
                            text=True, capture_output=True)
    print(result.stdout or result.stderr)
if not (path / "scripts" / "live_smoke.py").is_file():
    raise SystemExit("scripts/live_smoke.py missing: check REPO_URL and the branch")

os.chdir(REPO_DIR)
print("cwd:", os.getcwd())
print(subprocess.run(["git", "-C", REPO_DIR, "log", "-1", "--oneline"],
                     text=True, capture_output=True).stdout.strip())


## 4. Secrets (optional)

Semantic Scholar is opt-in and stays off for the resume demo: the S2 key is only needed
for the metadata/references pass, and turning it on makes each publication depend on a
second rate-limited API. When `USE_S2` is true the key is read from Colab **Secrets**
(key icon in the left sidebar) and never printed.


In [ ]:
import getpass

S2_READY = False
if USE_S2:
    try:
        from google.colab import userdata

        os.environ["SEMANTIC_SCHOLAR_API_KEY"] = userdata.get("SEMANTIC_SCHOLAR_API_KEY")
    except Exception:
        if not os.environ.get("SEMANTIC_SCHOLAR_API_KEY"):
            entered = getpass.getpass("SEMANTIC_SCHOLAR_API_KEY: ")
            if entered:
                os.environ["SEMANTIC_SCHOLAR_API_KEY"] = entered
    S2_READY = bool(os.environ.get("SEMANTIC_SCHOLAR_API_KEY"))
    print("semantic_scholar_key_set:", S2_READY)
else:
    print("Semantic Scholar disabled (arXiv-only smoke)")


## 5. Helper — run the smoke runner and show its output

Every step below calls the same CLI a developer would use locally, so the notebook is a
thin wrapper: nothing is re-implemented in a cell.


In [ ]:
import shlex
import subprocess


LAST_RUN = {}


def smoke(*args):
    """Run scripts/live_smoke.py from the repo root and echo its output."""
    command = [sys.executable, "-B", "-m", "scripts.live_smoke", *map(str, args)]
    print("$", " ".join(shlex.quote(part) for part in command), "\n")
    process = subprocess.run(command, cwd=REPO_DIR, text=True, capture_output=True)
    print(process.stdout)
    if process.stderr.strip():
        print("stderr:", process.stderr.strip())
    print("exit_code:", process.returncode)
    global LAST_RUN
    LAST_RUN = dict(command=" ".join(command), exit_code=process.returncode,
                    stdout=process.stdout, stderr=process.stderr)
    return process.returncode


def common():
    args = ["--pilot", PILOT, "--batch-id", BATCH_ID,
            "--corpus-root", CORPUS_ROOT, "--log-root", LOG_ROOT,
            "--report-root", REPORT_ROOT, "--max-ids", "10"]
    if BENCHMARK:
        # Same selection and checkpoints; adds wall time, RAM and disk.
        args.append("--benchmark")
    return args


print("helpers ready")


## 6. Smoke test, interrupted after paper 5

`--interrupt-after 5` raises the same `KeyboardInterrupt` that Ctrl+C raises, from the
Batch progress callback, after the fifth paper's snapshot is durable. That makes the
interruption reproducible while exercising the real recovery path.

To interrupt **by hand** instead: start the same command without `--interrupt-after`, then
click the stop button (or `Runtime > Interrupt execution`) while paper 5 is running, and
jump to step 7. The result is the same: an attempt whose status is `interrupted`, with the
finished papers already checkpointed on Drive.


In [ ]:
code = smoke("smoke", *common(), "--interrupt-after", INTERRUPT_AFTER)
print()
print("Expected: the run stops after paper", INTERRUPT_AFTER,
      "and prints the resume hint. exit_code 0 means the interruption was recorded.")


## 7. Inspect the interrupted state

Two things must be true before the resume is meaningful:

- the first attempt is on disk with status `interrupted`, and the finished papers are marked
  `processed`;
- no `.tmp`/`.part` file was left behind by the interrupted writer.


In [ ]:
import json

batch_dir = Path(LOG_ROOT) / BATCH_ID
attempt_paths = sorted((batch_dir / "attempts").glob("*.json"))
print("batch dir :", batch_dir, "| exists:", batch_dir.is_dir())
print("attempts  :", len(attempt_paths), "log(s)")
print("last run  :", LAST_RUN.get("command", "(none in this session)"))
print("exit_code :", LAST_RUN.get("exit_code"))
if LAST_RUN.get("stderr", "").strip():
    print("stderr    :", LAST_RUN["stderr"].strip()[-1500:])
if not attempt_paths:
    raise SystemExit(
        "No attempt log was written, so there is nothing to inspect. Read the "
        "exit_code/stderr above: the smoke cell (step 6) is the one that failed. "
        "Common causes: BRANCH still \"main\" (clone brings no live_smoke.py), "
        "or DRIVE_BASE still pointing at a batch that already exists."
    )
attempts = sorted(
    (json.loads(p.read_text(encoding="utf-8")) for p in (batch_dir / "attempts").glob("*.json")),
    key=lambda value: value["attempt_number"],
)
latest = attempts[-1]
print("attempt:", latest["attempt_number"], "status:", latest["status"])
for row in latest["papers"]:
    print(f"  {row['index'] + 1:>2}. {row['arxiv_id']} state={row['state']}")

done = sum(1 for row in latest["papers"] if row["state"] == "processed")
print("\npapers finished before the interruption:", done)

leftovers = [p for p in Path(DRIVE_BASE).rglob("*")
             if any(marker in p.name for marker in (".tmp", ".part", ".writing"))]
print("leftover temp files:", len(leftovers))
for path in leftovers[:10]:
    print("  ", path)


## 8. Resume — skip what is already crawled

The same `--batch-id`, corpus root and selection are required; `resume=True` is what the
CLI adds. Batch re-validates every checkpoint with its SHA-256 first, then Pipeline continues
from the first unfinished paper. Papers 1-5 are answered from the discovery cache and their
versions are skipped, so they cost no request.


In [ ]:
code = smoke("resume", *common())
print()
print(f"Expected: 'Status: finished'. Papers 1-{INTERRUPT_AFTER} show discovery=cached and "
      "source actions=skipped; the rest are queried and downloaded.")


## 9. Corpus integrity (contract v2)

Independent of the logs: re-reads every paper, re-hashes every kept file and checks that a
version folder holds nothing but `.tex`/`.bib`.


In [ ]:
if LAST_RUN.get("exit_code") not in (0, None):
    print("WARNING: the last runner call exited "
          f"{LAST_RUN['exit_code']}; verify below shows its consequences.")
smoke("verify", "--pilot", PILOT,
      "--corpus-root", CORPUS_ROOT, "--log-root", LOG_ROOT,
      "--json-out", f"{LOG_ROOT}/{BATCH_ID}/evidence/corpus-verification.json")


## 10. Resume proof — the artifact for sign-off

Compares the attempt logs with each other and with the pre-resume snapshot recorded in
step 6, then writes `resume-proof.md` next to the batch logs on Drive.


In [ ]:
if LAST_RUN.get("exit_code") not in (0, None):
    print("WARNING: the last runner call exited "
          f"{LAST_RUN['exit_code']}; proof below shows its consequences.")

code = smoke("proof", "--pilot", PILOT, "--batch-id", BATCH_ID,
             "--corpus-root", CORPUS_ROOT, "--log-root", LOG_ROOT)
proof_file = Path(LOG_ROOT) / BATCH_ID / "evidence" / "resume-proof.md"
print("proof written:", proof_file.is_file())


In [ ]:
from IPython.display import Markdown, display

display(Markdown(proof_file.read_text(encoding="utf-8")))


## 11. Optional — Semantic Scholar pass (separate batch)

Metadata and references are a separate, opt-in pass. It needs its own `batch_id`: a batch
definition cannot be changed after creation. Only run this after the smoke above passes.


In [ ]:
S2_BATCH = f"{BATCH_ID}-s2"
print("USE_S2:", USE_S2, "| key ready:", S2_READY)
if USE_S2 and S2_READY:
    smoke("smoke", "--pilot", PILOT, "--batch-id", S2_BATCH,
          "--corpus-root", CORPUS_ROOT, "--log-root", LOG_ROOT,
          "--max-ids", "10", "--s2",
          "--s2-min-interval", S2_MIN_INTERVAL)
else:
    print("Skipped. Set USE_S2 = True in step 0 and add SEMANTIC_SCHOLAR_API_KEY to run it.")


## 12. Evidence to hand over

Copy these paths into the M1-T02 report / Trello card:

| Evidence | Path |
| --- | --- |
| Resume proof (PASS/FAIL) | `logs/<batch-id>/evidence/resume-proof.md` |
| Corpus verification | `logs/<batch-id>/evidence/corpus-verification.json` |
| Attempt 1 (interrupted) | `logs/<batch-id>/attempts/<attempt-id>.json` |
| Attempt 2 (resume) | `logs/<batch-id>/attempts/<attempt-id>.json` |
| Operator view per run | `logs/<batch-id>/evidence/*-smoke.json`, `*-resume.json` |
| Corpus | `corpus/<folder-key>/{metadata,references,manifest}.json` + `v1/`, `v2/` |

Sign-off checklist for Toan:

- [ ] `proof` exits 0 and prints `Result: PASS`.
- [ ] Attempt 1 status `interrupted`, attempt 2 status `finished`.
- [ ] Papers 1-5 show `discovery=cached` and no re-download.
- [ ] `verify` reports 0 problems and 0 leftover temp files.
- [ ] Every `vN/` folder holds only `.tex`/`.bib`.

Then the full crawl can be authorised with a new `batch_id`.


In [ ]:
batch_dir = Path(LOG_ROOT) / BATCH_ID
for path in sorted(batch_dir.joinpath("evidence").glob("*")):
    print(path)
print("\npapers on Drive:", len(list(Path(CORPUS_ROOT).glob("[0-9]*"))))


In [ ]:
import json

batch_dir = Path(LOG_ROOT) / BATCH_ID
evidence = batch_dir / "evidence"
proof_path = evidence / "resume-proof.md"
verify_path = evidence / "corpus-verification.json"

def _safe_json(target):
    try:
        return json.loads(target.read_text(encoding="utf-8"))
    except (OSError, ValueError):
        return None


attempts = sorted(
    (json.loads(p.read_text(encoding="utf-8")) for p in (batch_dir / "attempts").glob("*.json")),
    key=lambda value: value["attempt_number"],
)
bundle = {
    "batch_id": BATCH_ID,
    "git": {
        "url": REPO_URL,
        "branch": BRANCH,
        "commit": subprocess.run(["git", "-C", REPO_DIR, "log", "-1", "--oneline"],
                                 text=True, capture_output=True).stdout.strip(),
    },
    "runtime": {
        "python": sys.version.split()[0],
        "platform": platform.platform(),
        "drive_mounted": MOUNTED,
        "s2_enabled": bool(USE_S2),
    },
    "proof": proof_path.read_text(encoding="utf-8") if proof_path.is_file() else None,
    "verification": (json.loads(verify_path.read_text(encoding="utf-8"))
                     if verify_path.is_file() else None),
    "benchmark_reports": [
        {
            "experiment_id": experiment.name,
            "batch_status": experiment_value.get("batch_status"),
            "execution_status": experiment_value.get("execution_status"),
            "wall_seconds": ((experiment_value.get("resource_metrics") or {})
                             .get("metrics_v2") or {}).get("wall_seconds"),
            "peak_ram_mb": (experiment_value.get("resource_metrics") or {})
                           .get("peak_ram_mb"),
            "ram_source": (experiment_value.get("resource_metrics") or {})
                          .get("ram_source"),
        }
        for experiment in sorted(Path(REPORT_ROOT).glob("*"))
        for experiment_value in [_safe_json(experiment / "report.json")]
        if isinstance(experiment_value, dict)
    ],
    "attempts": [
        {
            "attempt_number": item["attempt_number"],
            "attempt_id": item["attempt_id"],
            "status": item["status"],
            "started_at": item["started_at"],
            "ended_at": item["ended_at"],
            "previous_incomplete_attempts": item.get("previous_incomplete_attempts"),
            "papers": [
                {
                    "index": row["index"],
                    "arxiv_id": row["arxiv_id"],
                    "state": row["state"],
                    "discovery": ((row.get("result") or {}).get("discovery") or {}).get("action"),
                    "source": {
                        name: record.get("action")
                        for name, record in
                        ((row.get("result") or {}).get("source") or {}).items()
                    },
                    "paper_status": (row.get("result") or {}).get("paper_status"),
                }
                for row in item["papers"]
            ],
        }
        for item in attempts
    ],
}

target = Path("/content/evidence-bundle.json")
target.write_text(json.dumps(bundle, ensure_ascii=False, indent=1), encoding="utf-8")
drive_copy = evidence / "evidence-bundle.json"
drive_copy.write_text(target.read_text(encoding="utf-8"), encoding="utf-8")
print("bundle:", target, round(target.stat().st_size / 1024, 1), "KiB")
print("also on Drive:", drive_copy)

print("\n--- paste everything below this line back into the chat ---")
print("PROOF_OK:", "Result: PASS" in (bundle["proof"] or ""))
print("VERIFY_OK:", bool(bundle["verification"] and bundle["verification"].get("ok")))
for report in bundle["benchmark_reports"]:
    print(f"BENCH {report['experiment_id'][:8]}: "
          f"batch={report['batch_status']} wall={report['wall_seconds']} "
          f"peak_ram_mb={report['peak_ram_mb']} ram_source={report['ram_source']}")
for item in bundle["attempts"]:
    processed = sum(1 for row in item["papers"] if row["state"] == "processed")
    print(f"ATTEMPT {item['attempt_number']}: status={item['status']} "
          f"processed={processed}/{len(item['papers'])}")
    for row in item["papers"]:
        print(f"  {row['index'] + 1:>2}. {row['arxiv_id']} state={row['state']} "
              f"discovery={row['discovery']} source={row['source']} "
              f"paper={row['paper_status']}")
